In [2]:
#LINK: https://github.com/several27/FakeNewsCorpus/tree/master?tab=readme-ov-file
import pandas as pd
from bertopic import BERTopic # importing BERTopic for topic modeling
# importing TFIDF & sklearns built-in english stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
#importing lemmatizer to reduce words to base form
import re
import string
from nltk.stem import WordNetLemmatizer

In [3]:
# reading in the junk science articles from a CSV file into a pandas data frame & display
df_junksci_data = pd.read_csv("junk_sci_articles.csv")
df_junksci_data.head()

,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary,source
0,351,414.0,collective-evolution.com,junksci,http://www.collective-evolution.com/2018/01/23...,"Decades ago, when an individual claimed to wor...",2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,The Man Who Blew The Lid off of AREA 51 Shows ...,NaN,NaN,[''],Bob Lazar is the man who blew the lid open on ...,NaN,NaN,NaN
1,354,417.0,collective-evolution.com,junksci,http://www.collective-evolution.com/2018/01/24...,"Hey, you’re here, you made it! Congratulations...",2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Starbucks & McDonald’s Take Huge Steps Towards...,NaN,NaN,[''],Do these companies really care about the envir...,NaN,NaN,NaN
2,355,418.0,collective-evolution.com,junksci,http://www.collective-evolution.com/2018/01/24...,Building your network is key to a meaningful a...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Networking Is Now As Easy As Swiping Right,NaN,NaN,[''],Building your network is key to a meaningful a...,NaN,NaN,NaN
3,618,701.0,collective-evolution.com,junksci,http://www.collective-evolution.com/2018/01/20...,Disease has been on the rise for multiple deca...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,"Autophagy, Fasting & Exercise: Scientist Revea...",NaN,NaN,[''],Disease has been on the rise for multiple deca...,NaN,NaN,NaN
4,619,702.0,collective-evolution.com,junksci,http://www.collective-evolution.com/2018/01/19...,Chickens on sale for human consumption in Brit...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Chickens From British Supermarket Show Record ...,NaN,NaN,[''],Chickens on sale for human consumption in Brit...,NaN,NaN,NaN


In [4]:
# number of rows/articles in the dataframe
len(df_junksci_data)

17408

In [5]:
# displaying the dataframe with only the content column (this is where the articles are)
junk_sci_content = df_junksci_data[["content"]]
junk_sci_content

,content
0,"Decades ago, when an individual claimed to wor..."
1,"Hey, you’re here, you made it! Congratulations..."
2,Building your network is key to a meaningful a...
3,Disease has been on the rise for multiple deca...
4,Chickens on sale for human consumption in Brit...
...,...
17403,"Stanley Popovich, Contributing Writer\n\nWakin..."
17404,(Natural News) Don’t get preppers wrong. They’...
17405,Beating cancer -- how to take charge of your c...
17406,"(Natural News) In 20 years, humans will have r..."


In [6]:
# taking out custom stop words which are common words manually - (did it "iteratively" )
custom_words = list(set([
    'like','years','new','use','com','just','people','time','make','way','study','said','reference','world','life','news','state',
    'public','year','water','government','human','body','children','foods','free','health','food','work','know','things','energy',
    'offers','ingredients','database','levels','risk','industry','media','org','looking','right','truth','global','creation','earth',
    'benefits','online','medical','help','want','think','power','mind','times','need','love','don','site','dr','lists','programs',
    'products','healthy','vitamin','diet','vaccine','vaccines','www','used','evolutionary','researchers','research','longer','areas',
    'posted','available','page','percent','high','states','control','information','plant','natural','mothers','healing','documentaries',
    'medicines','herbs','light','article','planet','great','god','self','feel','day','university','cell','effects','patients','brain',
    'cells','studies','dice','thomas','fossil','company','money','drug','really','prevent','sugar','try','eating','diseases','oil',
    'nutrition','monsanto','big','gmo','age','scientists','history','million','000','icr','evidence','flood','person','say','ve',
    'good','search','searching','interested','powered','center','series','conditions','environment','based','going','caused','heart',
    'organic','blood','cause','chemical','flu','http','companies','according','chemicals','toxic','fda','naturalnews','reality',
    'physical','change','goodgopher','experience','different','nature','consciousness','dna','old','ago','evolution','fossils','today',
    'long','did','science','solar','sun','watch','weather','geoengineeringwatch','wigington','geoengineering','dane','climate',
    'recommended','women','alternative','non','medicine','treatment','treat','drugs','cancer','nutritional','better','important',
    'tea','content','best','milk','skin','raw','eat','including','lead','breast','known','symptoms','doctors','immune','exposure',
    'published','animals','ice','data','radiation','humans','institute','called','genes','engineering','species','does','true','come',
    'spiritual','man','believe','universe','ancient','cover','america','consumer','profit','organization','tell','education','supply',
    'dangerous','freedom','process','problem','country','number','place','word','source','fact','publishing','website','sure','home',
    'downloadable','resource','guide','expectant','investigation','offering','nonprofit','contain','original','large','extent',
    'expressed','saying','share','basic','concept','month','care','challenge','conventional','actually','author','recent','case',
    'group','result','report','special','solution','interaction','applied','provided','link','copy','html','clickable','plus',
    'newsletter','daily','subscribe','editor','indcom','ind','introvert','enjoy','similar','journey','entire','rewarding','coaching',
    'educating','chopped','basil','mint','honey','therapy','candle','surgery','topic','library','chart','honest','dare','phytonutrients',
    'pulitzer','salad','atlanta','email','encouraged','pinterest','fan','wwwfacebookcompagesthriveliving','wwwtwittercomthriveliving',
    'twitter','microwave','fusion','fluoridated','bodymindsoulspirit','google','archive','material','recommendation','advice','licensed',
    'practitioner','responsibility','commentary','wellness','grant','exposing','plantbased','minidocumentaries','download','cartoon',
    'view','nutrientreferencecom','healingfoodreferencecom','webseedcom','form','family','look','waking','living','solid','idea',
    'consider','example','alan','watt','hour','address','enter','talk','far','normal','thinking','matter','beginning','pretty',
    'physic','cutting','edge','comprehensive','plan','improve','enhance','achieve','act','real','story','city','question','video',
    'law','american','thank','personal','lifestyle','subscriber','holiday','sole','professional','opinion','protected','speech',
    'sell','term','founder','individual','rbgh','magma','dandelion','increase','bacteria','fruit','protein','cold','thanksgiving',
    'appear','ask','oops','order','meant','javascript','bitcoin','phillip','intuition','targeted','supplementation','loss','kit',
    'remedy','herbal','prepare','cabnet','versatile','minor','firstaid','xylitol','godly','gift','nostradamus','repeat','reliving',
    'palahniuk','police','stuck','threat','chuck','frankincense','lucy','tsa','alex','beer','jersey','development','bubble','test',
    'substitute','disclaimer','reserved','trademark','pharmacist','doctor','herb','nutrient','medication','pain','pile','sandstone',
    'dung','bird','wakefield','oneinamillion','happier','selfdirected','writing','thousand','solve','spends','acid','reflux','kimmel',
    'matrix','revealed','feathered','ohio','jupiter','spider','rappoport','jon','helium','week','imagine','pollution','aphid','death',
    'kefir','toilet','dumpster','rock','political','ocean','morris','obama','war','dinosaur','president','geoengineeringwatchorg',
    'trump','simple','stocking','philosophy','grief','ride','jayme','durant','soul','friend','modified','let','thought','fluoride',
    'vibration','meditation','feeling','meat','anderson','chimp','tomkins','epigenetic','genome','selection','pseudogenes','pluto',
    'finch','rna','omega','juice','vegetable','coconut','counterthink','rutherfordorg','despite','whitehead','republished','switch',
    'promise','future','monster','john','flint','giza','hunt','quantum','pyramid','inca','rapeseed','box','anna','wee','voice',
    'whisper','conscious','proof','sovereignty','alert','oral','contraceptive','visit','educational','hard','usage','llc','construed',
    'assumes','breastmilk','liberal','gunpoint','microbiome','gulf','prescott','gregg','naturalnewscom','daiichi','seafloor','sediment',
    'disaster','nuclear','adam','tepco','cesium','reactor','radioactive','fukushima','aspect','beautiful','sound','allows','major',
    'piano','sonata','quality','sounds','ministry','moringa','journal','school','include','pesticide','reported','using','rate',
    'type','disease','asthma','nodule','anunnaki','disabled','lorio','reenable','appears','flickr','reading','length','goal','optimal',
    'update','set','circumstance','effort','invisible','edition','market','latest','consultant','researcher','modification','la',
    'cat','marijuana','cannabinoids','suggests','needed','commercial','seed','cmb','diamond','mutation','butter','entirely','misunderstood',
    'dairy','product','contains','touted','super','margarine','lyme','conspiracy','issue','possible','reason','common','point','mean',
    'live','technology','privacy','network','purpose','depletion','deficiency','healthcare','registered','servicemarks','agreement',
    'entertainment','audience','spin','creative','explosive','collection','weekly','seat','mount','helen','guava','facebook','passed',
    'legislation','freeman','regulation','executive','canola','action','support','linking','permalink','cite','follow','embed','gorski',
    'msg','dog','broccoli','extreme','aluminum','population','syrup','cow','warfare','hurricane','listen','mozart','connection',
    'deeper','moment','dimensional','injury','healer','chakra','backwards','sponge','miso','melanosomes','deodorant','eliminator',
    'megasequences','lemon','summary','samantha','wwwpinterestcomthriveliving','wwwgooglceziyr','wwwthrivelivingnet','wwwnaturalnewscomauthorhtml',
    'wwwnaturalnewscomtermsshtml','spiritfoods','household','image','coverage','brian','nasa','noah','henry','feature','surface',
    'secular','biblical','dolgin','justh','equifax','bank','banking','currency','loan','payment','kathy','painting','euro','tpp',
    'outdoors','farm','small','described','newsweek','statin','cleveland','ebola','pharmaceutical','property','agency','exercise',
    'green','united','read','damage','supplement','status','saturated','joint','confused','premature','ida','superfood','villain',
    'theory','evolutionist','genesis','phd','sea','discovery','layer','origin','bone','creature','fibrillation','increased','low',
    'atrial','growing','zombie','brand','signature','canadian','slogan','military','ongoing','harvey','december','contrail','assault',
    'structure','lionel','firestorm','gyre','animated','backyard','smoothie','sayer','pigeon','artemisinin','pastor','hit','chance',
    'cdc','functional','leaf','fresh','phone','parent','triclosan','emf','baby','wireless','wifi','electromagnetic','emp','angiosperm',
    'coach','coronary','chest','saudi','cut','bypass','warmingclimate','vaxxed','stinging','danish','thorsen','nettle','swamp',
    'tiahuanaco','camu','astaxanthin','punku','puma','convinced','ready','anxiety','smith','stock','spend','ethan','nicotine','ashwagandha',
    'firewood','wisdom','near','ranked','sovereign','latenight','cornered','pauling','icahn','relationship','star','alien','dream',
    'spirit','humanity','stone','civilization','moon','space','herbert','hematoma','affected','bitcoins','ley','limited','doing',
    'rise','premium','kratom','father','child','scripture','christian','church','bible','christ','lord','eightlegged','annihilated',
    'oven','spewing','replete','drafting','opiate','monocrop','grow','mcgee','medicinal','herbreferencecom','antibiotic','weight',
    'fat','presence','awareness','danger','goodgophercom','owner','misuse','promoter','recommending','courtesy','paste','emailed',
    'ownership','respective','healthway','store','deli','bread','especially','productivity','service','cost','insurance','demand',
    'minimum','issued','failing','relief','receive','taking','claim','past','book','event','scientific','able','kind','end','unaware',
    'laptop','boy','sequence','tissue','preparation','winter','prewired','shower','teenager','risktaking','breastfed','confiscation',
    'appendix','field','vet','list','thing','hand','having','simply','crop','national','federal','course','getting','little','lot',
    'general','came','causing','web','indicates','intended','limitation','cooperation','consideration','imply','intention','selfsabotage',
    'hear','wasn','quite','performed','pastured','created','eye','modern','team','fish','discovered','tree','function','pulverized',
    'delightfully','corrective','maximized','hunter','battle','maker','november','primary','spraying','catastrophe','geoengineered',
    'exclusively','gun','diabetes','gut','coffee','legalization','psychological','reveals','party','battery','leftist','shouldn',
    'pipeline','newstarget','night','spoken','got','proverbial','zen','driving','main','aromatherapy','recipe','cup','apple',
    'vinegar','hospital','ravensthorpe','constitution','wasn','pastured','clinic','notice','gland','thyroid','arthritis','making',
    'written','likely','higher','instead','continue','given','finding','recently','activity','earns','begin','mentioned','reader',
    'code','represents','theyre','click','wish','favorite','balanced','troublesome','famous','making','night','got','wasn','special',
    'simply','likeminded','vet','written','general','treadmill','primary','warming','catastrophe','november','maker','spraying','boost',
    'predict','ultimate','admits','resulting','congress','society','crop','food','team','function','discovered','writer','impact',
    'airborne','main','current','optimum','phil','wilson','barack','kennedy','mark','isaac','vic','carolanne','jimmy','bezos',
    'elon','maria','shiva','sitchin','kepler','makia','marco','torres','dylan','anna','jon','pietrowski','sarich','catherine',
    'gregg','newall','brinker','pronsky','shi','qin','huang','mar','navajo','emperor','offit','craig','hebert','sanford','alex',
    'lucy','holmes','goodman','fassa','jesus','christina','sherwin','jockers','bardot','wolters','kluwer','icahn','pauling',
    'carolanne','bach','gates','bundrant','donald','llp','katherine','comey','mcdonald','thorsen','mcgee','kratom','wolf','keen',
    'likeminded','carol','michael','samsel','jake','hillary','clinton','sander','armstrong','campbell','dawkins','bushcraft',
    'preoperative','beard','peco','whats','got','wasn','noted','list','presented','manufacturer','specific','note','following',
    'international','attempting','frequent','morning','workout','joe','sprint','petroglyph','countless','anal','born','congress',
    'dietary','routine','narrative','familiar','worldwide','peco','method','massively','galaxy','orbit','think','object','built','found',
    'glutamic','bug','ability','nation','official','california','produce','environmental','branch','economy','coming','presidential',
    'election','macroevolution','considered','note','protecting','compassion','democratic','democrat','tool','veteran','rico','puerto',
    'temperature','expose','wildfire','deception','october','cbd','inherent','fluctuating','tornadic','peanut','mitochondrial',
    'ceremony','allergy','shown','expert','caffeine','infection','essential','lower','chronic','cholesterol','antioxidant','stress',
    'reduce','liver','drink','ive','riverside','killifish','skill','detergent','ranger','mike','copyright','passing','chicken',
    'restaurant','pleiotropy','fake','flagship','prey','wake','krauss','smoking','tobacco','smoker','model','learn','billion',
    'answer','away','belief','supplication','insect','pest','bee','policy','campaign','safety','epa','court','eclectic','comparison',
    'newstargetcom','eczema','trial','village','acre','plane','cane','overhead','pedestrian','birth','king','hotbutton','sumerian',
    'meme','cybersecurity','glucose','cyber','powerful','quark','probiotic','serotonin','potato','gluten','answer','intend','preterm',
    'handinhand','paradox','elixir','anytime','lowweight','china','forgiving','amyloid','karma','feather','implanting','busy','flare',
    'forest','venus','lung','ingredient','superfoods','dried','noncommercial','reprinting','delicious','sneezing','rage','butterflyshaped',
    'pay','thy','chef','diaspora','soil','thou','teotihuacan','germ','supercharge','acne','editing','adaptability','orphan','younger',
    'motive','strategic','procedure','bold','dominance','genetically','wheat','maca','isn','celiac','interview','radio','wiki','episode',
    'bang','table','challenging','illinois','lottery','unexplained','shrimp','derived','collagen','birch','schweitzer','homo','skull',
    'soft','chitin','cambrian','sjs','chicxulub','adhd','prozac','deceptive','middleeast','wildly','prone','drought','asd','effect',
    'farmer','certain','genetic','associated','mass','animal','air','factor','record','hawking','press','med','clin','economics',
    'phillipson','pdr','contraindication','avoid','clinisphere','opioid','optometry','resident','enormous','soy','mammogram','diverse',
    'transformed','roundup','choice','florida','depression','house','irma','rule','representative','imagining','abuse','horrific',
    'veto','tablespoon','copper','manganese','teaspoon','flour','pepper','aloe','cinnamon','cider','turmeric','ghana','learned',
    'sigmund','aluminium','vienna','malaria','cure','envious','arabia','successfully','coli','telescope','lymphatic','chimpanzee',
    'martian','calendula','couch','gilbert','balm','valerian','dreaded','gardener','greenland','senator','immigrant','fbi','cwc',
    'exemption','amber','western','thc','chemotherapy','vaccinated','virus','mmr','autism','gardasil','hemp','hpv','cannabis',
    'sending','mom','extrovert','adopt','curcumin','infant','haarp','heard','finnish','breastfeeding','serpent','methylation',
    'pterosaur','restriction','acknowledged','necessity','lupus','tetrapod','calcification','jbbardot','bardots','recedes','aurora',
    'homeopath','river','lingas','kidney','connecting','geoglyphs','embrace','preconception','picchu','nutritionist','polio',
    'machu','tsu','york','kennesaw','fsa','aeronautics','exodus','asthmatic','mood','chiropractic','beat','louse','breathing',
    'fragrant','likelihood','desired','delivering','renowned','weigh','impairment','infographic','lb','kraft','message','student',
    'practice','business','pharma','post','fear','mental','rbghfree','forrester','hatred','told','automatically','allopathic',
    'holistic','drugfree','nibiru','gmos','naturopath','ointment','alzheimer','eucalyptus','menopause','chamomile','sore','myopia',
    'pomegranate','moth','flush','chore','towel','clogging','rhinitis','plumber','planned','abortion','parenthood','iceland',
    'half','giving','illness','provides','citizen','killing','willing','circulation','fewer','url','al','greatest','venezuela', 
    'inflammation', 'black', 'mineral', 'muscle', 'enzyme','doesnt', 'white', 'local', 'nut', 'dollar'
]))

# Topic Modeling: BERTopic

In [7]:
# combining english stopwords + custom stop words as a list
all_stop_words = list(text.ENGLISH_STOP_WORDS) + list(custom_words) # sklearn's + custom words together
lemmatizer = WordNetLemmatizer() # creating a lemmatizer instance

# PREPROCESSING
def preprocess(text):
    text = str(text).lower() # converting text to lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # remove urls
    text = re.sub(r'<.*?>', '', text) # remove HTML tags
    text = re.sub(r'\d+', '', text) # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation)) # remove punctuation
    
    words = text.split() # tokenize on whitespace
    words = [lemmatizer.lemmatize(word) for word in words # lemmatize each token
             if word not in all_stop_words and len(word) > 2] # keep words only that are length > 2
    return " ".join(words) # join back to a cleaned after string

# 1. convert 'content' column to a list of text documents
documents = junk_sci_content['content'].tolist() # extracting article text as a python list

# 2. clean all documents
documents_cleaned = [preprocess(doc) for doc in documents] # apply preprocessing to every document

# 3. using TfidfVectorizer as the vectorizer_model for BERTopic
vectorizer_model = TfidfVectorizer(
    stop_words = all_stop_words, # applying stopwords at vectorization
    min_df = 2,   # ignore terms that appear in fewer than 2 articles
    max_df = 0.9  # ignore terms that appear in more than 90% of articles
)

# 4. create and fit BERTopic model (31 main topics) - actually 30 as -1 is not counted
topic_model = BERTopic(
    vectorizer_model = vectorizer_model, # using TF-IDF
    verbose = True,
    nr_topics = 51
)
# 5. fitting the model and getting topic assignment per doc
topics, probabilities = topic_model.fit_transform(documents_cleaned)

# 6. print top 10 non-empty words for topics 0–30
for topic_num in range(51): # looping over topic id's from 0 - 30
    if topic_num in topic_model.get_topics(): # check if topic exists after reduction
        words = [w for w, _ in topic_model.get_topic(topic_num) if w.strip()] # get the words for this topic
        print(f"Topic {topic_num}: ", words[:10]) # print first 10 words for that topic

# 7. print outlier / noise topic (-1), if it exists
if -1 in topic_model.get_topics(): # checking if BERTopic created the outlier
    outlier_words = [w for w, _ in topic_model.get_topic(-1) if w.strip()] # get its words & remove empty strings 
    print("Topic -1 (outliers):", outlier_words[:10]) # printing first 10 outlier-topic words


2025-11-26 14:54:38,635 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 544/544 [00:37<00:00, 14.54it/s]
2025-11-26 14:55:18,207 - BERTopic - Embedding - Completed ✓
2025-11-26 14:55:18,207 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-26 14:55:27,380 - BERTopic - Dimensionality - Completed ✓
2025-11-26 14:55:27,381 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` 

Topic 0:  ['bike', 'matt', 'appearing', 'calm', 'leave', 'position']
Topic 1:  ['magnesium', 'yoga', 'chocolate', 'selenium', 'disorder', 'calcium', 'sleep', 'effective', 'lavender', 'cigarette']
Topic 2:  ['mascot', 'contest', 'award', 'outsmarted', 'progressivism', 'coyote', 'smiths', 'roulette', 'dynamite', 'blew']
Topic 3:  ['debt', 'atmospheric', 'methane', 'obamacare', 'earthquake', 'terrorist', 'storm', 'aerosol', 'financial', 'cooldown']
Topic 4:  ['glyphosate', 'plastic', 'salmon', 'herbicide', 'mosquito', 'zika', 'monsantos', 'toxin', 'agriculture', 'residue']
Topic 5:  ['dating', 'sedimentary', 'evolved', 'decay', 'organism', 'marine', 'gene', 'core', 'design', 'mammal']
Topic 6:  ['gods', 'icrs', 'emotion', 'creator', 'dont', 'teaching', 'partner', 'understand', 'ego', 'blessing']
Topic 7:  ['cereal', 'glutenfree', 'subway', 'mcdonalds', 'sauce', 'cheese', 'menu', 'sandwich', 'burger', 'babe']
Topic 8:  ['magnetic', 'ufo', 'astronomer', 'earths', 'mercury', 'lunar', 'jupite

In [8]:
# adding the output above in a data frame
rows = []
for topic_num in range(51):
    if topic_num in topic_model.get_topics():
        words = [w for w, _ in topic_model.get_topic(topic_num) if w.strip()]
        rows.append({
            "Topic": topic_num,
            "Top_Words": ", ".join(words[:10])
        })

# include outlier topic (-1) if present
if -1 in topic_model.get_topics():
    outlier_words = [w for w, _ in topic_model.get_topic(-1) if w.strip()]
    rows.append({
        "Topic": -1,
        "Top_Words": ", ".join(outlier_words[:10])
    })

topics_df = pd.DataFrame(rows).sort_values("Topic").reset_index(drop=True)
display(topics_df)

,Topic,Top_Words
0,-1,"dont, processed, thats, level, effective, diso..."
1,0,"bike, matt, appearing, calm, leave, position"
2,1,"magnesium, yoga, chocolate, selenium, disorder..."
3,2,"mascot, contest, award, outsmarted, progressiv..."
4,3,"debt, atmospheric, methane, obamacare, earthqu..."
5,4,"glyphosate, plastic, salmon, herbicide, mosqui..."
6,5,"dating, sedimentary, evolved, decay, organism,..."
7,6,"gods, icrs, emotion, creator, dont, teaching, ..."
8,7,"cereal, glutenfree, subway, mcdonalds, sauce, ..."
9,8,"magnetic, ufo, astronomer, earths, mercury, lu..."


In [10]:

# mapping each row of topic words to a topic label
topic_labels = {
    -1: "Generic processing / noise",
     0: "Calm biking / physical activity",
     1: "Supplements, sleep, and relaxation",
     2: "Political satire / mascot campaign",
     3: "Climate, debt, and geopolitical crises",
     4: "Glyphosate, pesticides, and agriculture",
     5: "Evolution, fossils, and marine biology",
     6: "Religion, creationism, and emotion",
     7: "Fast food and processed meals",
     8: "Space, magnetism, and UFOs",
     9: "Vaccines and infectious disease",
    10: "Ancient Egypt and archaeology",
    11: "Natural health retailers and products",
    12: "Pregnancy, birth, and fertility",
    13: "Alzheimer’s and neurodegenerative disease",
    14: "Medicinal mushrooms and superbugs",
    15: "Smart meters, parasites, and nanotech",
    16: "Gender, beauty, and sexual behavior",
    17: "Pop song lyrics and love",
    18: "Homeschooling, kids, and dental/allergy issues",
    19: "Blood cells, oxygen, and circulation",
    20: "Retail giants and consumer stores",
    21: "Allergies and severe reactions",
    22: "Media guests and interviews",
    23: "Vision correction and pinhole glasses",
    24: "North Korean missiles and weapons",
    25: "Race, skin color, and primates",
    26: "Flights, aircraft, and cabin air",
    27: "Corporate biotech manipulation",
    28: "Lucid dreaming and hypnosis",
    29: "Chickens, coops, and homesteading",
    30: "Youth, crime, and violence statistics",
    31: "Political figures and certainty narratives",
    32: "Combat, gunfire, and battlefield scenes",
    33: "Religious music and scripture",
    34: "Breathing and prohibition language",
    35: "Laughter, happiness, and primate behavior",
    36: "Credit breaches and cybercrime",
    37: "Assault, rape, and severe events",
    38: "Fasting and Weizmann research",
    39: "Transgender surgery and regret",
    40: "Wildfires and vegetation burning",
    41: "Small events and debated significance",
    42: "Corn farming and agricultural markets",
    43: "Particle physics and subatomic particles",
    44: "Mechanisms, roles, and research analysis",
    45: "Skeptical questioning and argumentation",
    46: "Rabbit holes and writing metaphor",
    47: "Religious resins and incense",
    48: "Bovine hormones and rBST labeling",
    49: "Adaptogenic plants and tropical herbs",
}

topics_df["Topic_Name"] = topics_df["Topic"].map(topic_labels)
display(topics_df)


,Topic,Top_Words,Topic_Name
0,-1,"dont, processed, thats, level, effective, diso...",Generic processing / noise
1,0,"bike, matt, appearing, calm, leave, position",Calm biking / physical activity
2,1,"magnesium, yoga, chocolate, selenium, disorder...","Supplements, sleep, and relaxation"
3,2,"mascot, contest, award, outsmarted, progressiv...",Political satire / mascot campaign
4,3,"debt, atmospheric, methane, obamacare, earthqu...","Climate, debt, and geopolitical crises"
5,4,"glyphosate, plastic, salmon, herbicide, mosqui...","Glyphosate, pesticides, and agriculture"
6,5,"dating, sedimentary, evolved, decay, organism,...","Evolution, fossils, and marine biology"
7,6,"gods, icrs, emotion, creator, dont, teaching, ...","Religion, creationism, and emotion"
8,7,"cereal, glutenfree, subway, mcdonalds, sauce, ...",Fast food and processed meals
9,8,"magnetic, ufo, astronomer, earths, mercury, lu...","Space, magnetism, and UFOs"


### Past topic modeling done through LDA (OLD CODE) 

# LDA

In [8]:
"""
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction import text
import re
import string
from nltk.stem import WordNetLemmatizer

# Combine sklearn and custom stopwords
all_stop_words = set(text.ENGLISH_STOP_WORDS).union(set(custom_words))

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    # Remove URLs (http, https, www, etc.)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)               # Remove HTML tags
    text = re.sub(r'\d+', '', text)                 # Remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in all_stop_words and len(word) > 2]
    return " ".join(words)

# 1. Convert your 'content' column to a list of text documents
documents = junk_sci_content['content'].tolist()

# 2. Clean all documents
documents_cleaned = [preprocess(doc) for doc in documents]

# 3. Vectorize using TF-IDF and frequency filters
vectorizer = TfidfVectorizer(
    stop_words=list(all_stop_words),
    min_df=3,
    max_df=0.7,
)
doc_term_matrix = vectorizer.fit_transform(documents_cleaned)

# 4. Fit LDA and display topics
lda = LatentDirichletAllocation(n_components=25, random_state=42)
lda.fit(doc_term_matrix)
words = vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    print(f"Topic {idx}: ", [words[i] for i in topic.argsort()[-10:]])


"""

'\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.decomposition import LatentDirichletAllocation\nfrom sklearn.feature_extraction import text\nimport re\nimport string\nfrom nltk.stem import WordNetLemmatizer\n\n# Combine sklearn and custom stopwords\nall_stop_words = set(text.ENGLISH_STOP_WORDS).union(set(custom_words))\n\nlemmatizer = WordNetLemmatizer()\n\ndef preprocess(text):\n    text = str(text).lower()\n    # Remove URLs (http, https, www, etc.)\n    text = re.sub(r\'https?://\\S+|www\\.\\S+\', \'\', text)\n    text = re.sub(r\'<.*?>\', \'\', text)               # Remove HTML tags\n    text = re.sub(r\'\\d+\', \'\', text)                 # Remove numbers\n    text = text.translate(str.maketrans("", "", string.punctuation))\n    words = text.split()\n    words = [lemmatizer.lemmatize(word) for word in words if word not in all_stop_words and len(word) > 2]\n    return " ".join(words)\n\n# 1. Convert your \'content\' column to a list of text documents\nd